In [ ]:
from datasets import load_dataset
from model import CUR_DIR

In [17]:
# data_files = [
#     f"{CUR_DIR}/data/sft/everyday-conversations-llama3.1-2k/data/test_sft-00000-of-00001.parquet",
#     f"{CUR_DIR}/data/sft/everyday-conversations-llama3.1-2k/data/train_sft-00000-of-00001.parquet",
# ]


data_files = [

    f"{CUR_DIR}/data/sft/Magpie-Pro-300K-Filtered/data/train-00000-of-00003.parquet",
    f"{CUR_DIR}/data/sft/Magpie-Pro-300K-Filtered/data/train-00001-of-00003.parquet",
    f"{CUR_DIR}/data/sft/Magpie-Pro-300K-Filtered/data/train-00002-of-00003.parquet"
]


dataset = load_dataset("parquet", data_files=data_files)

Generating train split: 0 examples [00:00, ? examples/s]

In [36]:
type(dataset)

datasets.dataset_dict.DatasetDict

In [38]:
dataset

DatasetDict({
    train: Dataset({
        features: ['conversations', 'uuid'],
        num_rows: 300000
    })
})

In [22]:
len(dataset["train"])

300000

In [25]:
print(dataset["train"].column_names)

['conversations', 'uuid']


In [31]:
dataset["train"]["conversations"][0]

[{'from': 'human',
  'value': 'Can you explain the concept of equitable distribution in family law?'},
 {'from': 'gpt',
  'value': 'In family law, equitable distribution refers to the process of dividing marital assets and debts fairly and reasonably between spouses in the event of a divorce or separation. The goal of equitable distribution is to allocate the marital estate in a way that is just and equitable, considering the unique circumstances of each case.\n\nHere are some key principles and concepts related to equitable distribution:\n\n1. **Marital Property**: Marital property includes all assets and debts acquired during the marriage, regardless of whose name they are in. This can include real estate, bank accounts, investments, retirement accounts, vehicles, and personal property.\n2. **Separate Property**: Separate property, on the other hand, refers to assets and debts acquired before the marriage, inherited, or gifted to one spouse individually. These assets are typically no

In [ ]:
split = dataset["train"].train_test_split(test_size=0.1, seed=42)
train_ds, eval_ds = split["train"], split["test"]

In [34]:
type(train_ds)

datasets.arrow_dataset.Dataset

In [ ]:
eval_ds[0]

{'conversations': [{'from': 'human',
   'value': 'What are the benefits of using Keras for deep learning?'},
  {'from': 'gpt',
   'value': 'Keras is a popular, open-source, and widely-used deep learning library that provides an easy-to-use interface for building and deploying deep learning models. Here are some benefits of using Keras for deep learning:\n\n1. **Easy to Learn and Use**: Keras has a simple and intuitive API, making it easy for beginners to learn and use, even for those without extensive programming experience.\n2. **High-Level Abstraction**: Keras provides a high-level abstraction, allowing users to focus on building models rather than implementing low-level details. This makes it easier to experiment with different architectures and ideas.\n3. **Rapid Prototyping**: Keras enables rapid prototyping, allowing users to quickly build and test models, which is essential for deep learning research and development.\n4. **Modular Architecture**: Keras has a modular architecture

In [ ]:
type(eval_ds[0][0])

dict

In [ ]:
# <|im_start|>system
# You are a helpful assistant.<|im_end|>
# <|im_start|>user
# 什么是TCP三次握手？<|im_end|>
# <|im_start|>assistant
# TCP三次握手是TCP建立连接的过程...<|im_end|>

In [39]:
special_tokens = [
    "<|im_start|>",
    "<|im_end|>",
]

In [45]:
def apply_chat_template(messages):

    text = """<|im_start|>system
You are a helpful assistant.<|im_end|>
"""

    for message in messages:

        role_from = message["from"]
        role_value = message["value"]

        if role_from == "human":
            role = "user"
        elif role_from == "gpt":
            role = "assistant"
        else:
            raise ValueError(f"Unknown role: {role_from}")  

        text += f"<|im_start|>{role}\n"
        text += role_value
        text += "<|im_end|>\n"

    return text

In [46]:
apply_chat_template(eval_ds[0]["conversations"])

'<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\nWhat are the benefits of using Keras for deep learning?<|im_end|>\n<|im_start|>assistant\nKeras is a popular, open-source, and widely-used deep learning library that provides an easy-to-use interface for building and deploying deep learning models. Here are some benefits of using Keras for deep learning:\n\n1. **Easy to Learn and Use**: Keras has a simple and intuitive API, making it easy for beginners to learn and use, even for those without extensive programming experience.\n2. **High-Level Abstraction**: Keras provides a high-level abstraction, allowing users to focus on building models rather than implementing low-level details. This makes it easier to experiment with different architectures and ideas.\n3. **Rapid Prototyping**: Keras enables rapid prototyping, allowing users to quickly build and test models, which is essential for deep learning research and development.\n4. **Modular Architecture**: Ker

In [6]:
from tokenizers import Tokenizer
tokenizer = Tokenizer.from_file("/home/sllm_scratch/D2L/nsllm/model/tokenizer_fixed.json")
tokenizer.add_special_tokens(["<|im_start|>", "<|im_end|>"])
tokenizer.save("/home/sllm_scratch/D2L/nsllm/model/tokenizer_sft.json")

In [7]:
tokenizer = Tokenizer.from_file("/home/sllm_scratch/D2L/nsllm/model/tokenizer_sft.json")

tokenizer.encode("<|im_start|><|im_end|>").ids

[32000, 32001]

In [ ]:
import torch

from torch.utils.data import Dataset, DataLoader

class SFTDataset(Dataset):

    def __init__(self, dataset: Dataset, max_length=2048):
        
        self.dataset = dataset
        self.max_length = max_length
        self.tokenizer = tokenizer
        self.system_prompt = "You are a helpful assistant."

        self.buffer = []
        
        self.stats = {
            "total": 0,
            "invalid": 0,
            "user_too_long": 0,
            "too_long": 0,
            "kept": 0,
        }

        self._load()

    def _load(self):
        for idx in range(len(self.dataset)):

            self.stats["total"] += 1

            conversations = self.dataset[idx]["conversations"]
            if len(conversations) != 2:
                self.stats["invalid"] += 1
                continue

            from_human = conversations[0]["value"]
            from_gpt = conversations[1]["value"]

            user_text = f"<|im_start|>system\n{self.system_prompt}<|im_end|>\n<|im_start|>user\n{from_human}<|im_end|>\n"
            gpt_text = f"<|im_start|>assistant\n{from_gpt}<|im_end|>"
            

            user_ids = self.tokenizer.encode(user_text).ids
            gpt_ids = self.tokenizer.encode(gpt_text).ids

            if len(user_ids) >= self.max_length:
                self.stats["user_too_long"] += 1
                continue


            if len(user_ids) + len(gpt_ids) > self.max_length:
                self.stats["too_long"] += 1
                continue

            input_ids = user_ids + gpt_ids

            labels = [-100] * len(user_ids) + gpt_ids

            input_tensor = torch.tensor(input_ids, dtype=torch.long)
            label_tensor = torch.tensor(labels, dtype=torch.long)
                
            self.buffer.append((input_tensor, label_tensor))

            self.stats["kept"] += 1

        print(f"SFTDataset: {self.stats}")
            

    def __len__(self):
        return len(self.buffer)

    def __getitem__(self, index):
        return self.buffer[index]



class SFTCollator:

    def __init__(self, pad_token_id):
        self.pad_token_id = pad_token_id
    
    def __call__(self, batch):
        """
            batch 是 DataLoader 根据 batch_size=12 从 Dataset 中取出的 12 个样本组成的 list。
            batch = [[input_ids, label_ids], [input_ids, label_ids], ...]
        """
        # >>> batch = [[1, 'a'], [2, 'b']]
        # >>> c, d = zip(*a)
        # >>> c
        # (1, 2)
        # >>> d
        # ('a', 'b')

        input_ids, label_ids = zip(*batch)

        max_len = max(x.size(0) for x in input_ids)

        batch_input_ids = []
        batch_labels = []
        batch_attention_mask = []

        for ids, lbs in zip(input_ids, label_ids):
            pad_len = max_len - len(ids)

            batch_input_ids.append(
                torch.cat([
                    ids, 
                    torch.full((pad_len,), self.pad_token_id, dtype=torch.long)  # NOTE: 此处使用pad_token_id填充，此处要通过Embeddings层计算token的embedding， 所以需要使用pad_token_id的embedding
                ])
            )
            batch_labels.append(
                torch.cat([
                    lbs, 
                    torch.full((pad_len,), -100, dtype=torch.long)  # NOTE: 关键修正, label补齐长度时不要使用pad_token_id, 直接使用-100， 方便CrossEntropyLoss计算loss时ignore_index=-100的token
                ])  
            )

            batch_attention_mask.append(
                torch.cat([
                    torch.ones(
                        len(ids),
                        dtype=torch.long
                    ),
                    torch.zeros(
                        pad_len,
                        dtype=torch.long
                    )
                ])
            )
        
        return {
            "input_ids": torch.stack(
                batch_input_ids
            ),
            "labels": torch.stack(
                batch_labels
            ),
            "attention_mask": torch.stack(
                batch_attention_mask
            ),
        }




data_files = [

    f"{CUR_DIR}/data/sft/Magpie-Pro-300K-Filtered/data/train-00000-of-00003.parquet",
    # f"{CUR_DIR}/data/sft/Magpie-Pro-300K-Filtered/data/train-00001-of-00003.parquet",
    # f"{CUR_DIR}/data/sft/Magpie-Pro-300K-Filtered/data/train-00002-of-00003.parquet"
]


dataset_dict = load_dataset("parquet", data_files=data_files)

split = dataset_dict["train"].train_test_split(test_size=0.1, seed=42)
train_ds, eval_ds = split["train"], split["test"]


# train_ds = SFTDataset(train_ds)
sft_eval_ds = SFTDataset(eval_ds)

collator = SFTCollator(
    pad_token_id=tokenizer.token_to_id("<pad>")
)



eva_loader = DataLoader(
    sft_eval_ds,
    batch_size=12,
    num_workers=0,
    pin_memory=True,
    collate_fn=collator,
)

for item in eva_loader:
    input_ids = item["input_ids"]
    labels = item["labels"]
    attention_mask = item["attention_mask"]
    print(input_ids.shape)
    print(labels.shape)
    print(attention_mask.shape)
    break

SFTDataset: {'total': 10000, 'invalid': 0, 'user_too_long': 0, 'truncated': 16, 'kept': 10000}
torch.Size([12, 1624])
torch.Size([12, 1624])
torch.Size([12, 1624])


In [14]:
tokenizer.encode("<pad>").ids

[1]

In [17]:
labels[1, 10:300]

tensor([ -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100, 32000,  7475,   202,  4015,
          325,   261, 19705,   290,   266,  1412,  7176,    29,   202, 22751,
           67,   202,   224,   224,   224,   224,   224,   224,   224,   224,
          224,  3394, 30198, 13292,    14,   202,   224,   224,   224,   224,
          224,   224,   224,   224,   224,  1269,   224,  3929,   566,  6492,
          224,  1269,   202,   224,   224,   224,   224,   224,   224,   224,
          224,   224,  3394, 30198, 13292,    14,   202,   224,   224,   224,
          224,   224,   224,   224,   224,   224,   224,   224,   224,   224,
          224,   224,   224,   224,  1269,   202,   224,   224,   224,   224,
          224,   224,   224,   224,   224,   224,   224,   224,   224,   224,
          224,   224,   224,  1269,   202,   224,   224,   224,   224,   224,
          224,   224,   224,   224,   224,   224,   224,   224, 